# Análisis de Base de Datos: Aplicación de Libros

### Contexto del proyecto:
Durante la pandemia de COVID-19, con más personas en casa, 
las aplicaciones de lectura ganaron popularidad. Este análisis 
explora una base de datos de un servicio de libros para generar 
una propuesta de valor para un nuevo producto.

In [ ]:
# importar librerías
import pandas as pd
from sqlalchemy import create_engine

# conectar la base de datos

db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

### 1. Libros publicados después del 1 de enero de 2000

In [14]:
query = """
SELECT COUNT(*) 
FROM books
WHERE publication_date > '2000-01-01';
"""

pd.io.sql.read_sql(query, con=engine)

,count
0,819


Hay 819 libros publicados después del 1 de enero del 2000, 
lo que indica que el catálogo tiene una base sólida de títulos recientes.

### 2. Número de reseñas y calificación promedio por libro

In [ ]:


query = """
SELECT COUNT(*) 
FROM reviews
"""
pd.io.sql.read_sql(query, con=engine)


,count
0,2793


In [3]:
query = """
SELECT title, AVG(rating) 
FROM  ratings
INNER JOIN books
ON ratings.book_id = books.book_id
GROUP BY title
ORDER BY AVG(rating) DESC
LIMIT 10;

"""
pd.io.sql.read_sql(query, con=engine)

,title,avg
0,My Name Is Asher Lev,5.0
1,In the Hand of the Goddess (Song of the Liones...,5.0
2,Misty of Chincoteague (Misty #1),5.0
3,Neil Gaiman's Neverwhere,5.0
4,Stone of Farewell (Memory Sorrow and Thorn #2),5.0
5,Alas Babylon,5.0
6,Stolen (Women of the Otherworld #2),5.0
7,Evil Under the Sun (Hercule Poirot #24),5.0
8,The Ghost Map: The Story of London's Most Terr...,5.0
9,School's Out—Forever (Maximum Ride #2),5.0


Los libros mejor calificados no se concentran en un solo género 
ni época, sino que reflejan tendencias claras:

- Los lectores de géneros específicos como **fantasía, juvenil y misterio** 
  tienden a dejar reseñas muy positivas:
    - *In the Hand of the Goddess* (Song of the Lioness #2) — Tamora Pierce
    - *Stone of Farewell* (Memory, Sorrow, and Thorn #2) — Tad Williams
    - *School's Out—Forever* (Maximum Ride #2) — James Patterson

- Los **clásicos consolidados** mantienen un alto promedio:
    - *My Name Is Asher Lev* — Chaim Potok (1972)
    - *Alas, Babylon* — Pat Frank (1959)
    - *Evil Under the Sun* (Hercule Poirot #24) — Agatha Christie (1941)

Sin embargo, hay que aclarar que las calificaciones perfectas suelen corresponder a libros con pocas reseñas.
Esto sugiere que la popularidad y la calidad percibida no siempre van de la mano. Acá lo que se pretendía era mostrar la variedad de libros. 

### 3. Editorial con más libros (más de 50 páginas)

In [31]:
query = """
SELECT publisher, COUNT(book_id) AS num_books
FROM  publishers
INNER JOIN books
ON publishers.publisher_id = books.publisher_id
WHERE num_pages > 50
GROUP BY publisher
ORDER BY num_books DESC
LIMIT 1
"""
pd.io.sql.read_sql(query, con=engine)

,publisher,num_books
0,Penguin Books,42


La editorial Penguin Books lidera en publicaciones con más de 50 páginas, 
lo que la posiciona como el socio editorial más relevante para este servicio.

### 4. Autor con la calificación promedio más alta

In [30]:
query = """SELECT a.author, AVG(r.rating) AS avg_rating
           FROM authors a
           INNER JOIN books b
           ON a.author_id = b.author_id
           INNER JOIN ratings r
           ON b.book_id = r.book_id
           WHERE b.book_id IN (
                 SELECT book_id
                 FROM ratings
                 GROUP BY book_id
                 HAVING COUNT(*) >= 50
           )
           GROUP BY a.author
           ORDER BY avg_rating DESC
           LIMIT 1
"""
pd.io.sql.read_sql(query, con=engine)


,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.287097


La autora  J.K. Rowling tiene la calificación promedio más alta (4.287097) entre 
los autores con libros que cuentan con al menos 50 calificaciones, 
lo que lo convierte en una elección confiable para recomendar en la app.

### 5.Promedio de reseñas de usuarios muy activos

In [32]:
query = """
SELECT 
    AVG(num_reviews) AS avg_reviews
FROM (
    SELECT 
        r.username,
        COUNT(DISTINCT rev.review_id) AS num_reviews
    FROM ratings r
    LEFT JOIN reviews rev
        ON r.username = rev.username
    GROUP BY r.username
    HAVING COUNT(DISTINCT r.book_id) > 50
) t;
"""

pd.io.sql.read_sql(query, con=engine)

,avg_reviews
0,24.333333


Los usuarios más activos (más de 50 calificaciones) 
escriben en promedio 24.3 reseñas de texto. Esto sugiere que los usuarios 
más comprometidos también contribuyen significativamente con contenido escrito.

## Conclusiones:

- Hay **819 libros** publicados después del 1 de enero del 2000,
lo que indica que el catálogo cuenta con una base sólida de títulos recientes,
relevantes para lectores contemporáneos.

- Se calcularon las reseñas y calificación promedio para cada libro.
Varios títulos tienen una calificación perfecta de **5.0**, aunque es importante
notar que calificaciones perfectas suelen corresponder a libros con pocas reseñas.
Esto sugiere que la popularidad y la calidad percibida no siempre van de la mano.

- **Penguin Books** es la editorial con mayor número de publicaciones
con más de 50 páginas, con un total de **42 libros**. Esto la posiciona como
el socio editorial más relevante dentro de este servicio.

- **J.K. Rowling** (junto a Mary GrandPré) es la autora con la
calificación promedio más alta (**4.29**) entre los libros con al menos
50 calificaciones. Esto la convierte en una de las opciones más confiables
para recomendar dentro de la aplicación.

- Los usuarios más activos (más de 50 calificaciones) escriben
en promedio **24.33 reseñas** de texto. Esto indica que los usuarios más
comprometidos también contribuyen de forma significativa con contenido escrito,
siendo un segmento clave para la comunidad de la app.